In [ ]:
import pygame
import math
import sys

# 初始化 Pygame
pygame.init()

# 畫面設定
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Bouncing Ball in a Spinning Hexagon")
clock = pygame.time.Clock()

# 定義物理參數
GRAVITY = 0.5            # 重力加速度
FRICTION = 0.99          # 球碰撞後的摩擦（降低速度）
BALL_RADIUS = 10

# 球的初始狀態：位置、速度
ball_pos = pygame.Vector2(WIDTH // 2, HEIGHT // 4)
ball_vel = pygame.Vector2(3, 0)

# 六邊形參數
hex_center = pygame.Vector2(WIDTH // 2, HEIGHT // 2)
hex_radius = 200
rotation = 0             # 六邊形初始旋轉角度
spin_speed = 0.01        # 每次更新的旋轉角度

# 計算六邊形頂點（根據當前旋轉角度）
def get_hexagon_points(center, radius, rotation):
    points = []
    for i in range(6):
        angle = math.radians(60 * i) + rotation
        x = center.x + radius * math.cos(angle)
        y = center.y + radius * math.sin(angle)
        points.append(pygame.Vector2(x, y))
    return points

# 計算點到線段的距離以及最近點
def point_line_distance(point, line_start, line_end):
    line = line_end - line_start
    if line.length() == 0:
        return (point - line_start).length(), line_start
    t = max(0, min(1, (point - line_start).dot(line) / line.length_squared()))
    projection = line_start + t * line
    return (point - projection).length(), projection

# 主遊戲迴圈
running = True
while running:
    clock.tick(60)  # 60 FPS

    # 事件處理
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # 更新六邊形的旋轉角度
    rotation += spin_speed
    hex_points = get_hexagon_points(hex_center, hex_radius, rotation)

    # 更新球的物理狀態
    ball_vel.y += GRAVITY          # 加入重力
    ball_pos += ball_vel           # 更新位置

    # 檢查球與六邊形各邊的碰撞
    for i in range(6):
        # 每一條邊的起點和終點
        p1 = hex_points[i]
        p2 = hex_points[(i + 1) % 6]

        # 計算球到邊的最短距離及最近點
        dist, closest_point = point_line_distance(ball_pos, p1, p2)
        if dist < BALL_RADIUS:
            # 碰撞發生，計算碰撞法向量
            collision_normal = (ball_pos - closest_point).normalize()

            # 反射球的速度： v_new = v - 2(v·n)*n
            vel_dot_normal = ball_vel.dot(collision_normal)
            if vel_dot_normal < 0:  # 確保只對向內的碰撞做反射
                ball_vel = ball_vel - 2 * vel_dot_normal * collision_normal

                # 加入摩擦效果
                ball_vel *= FRICTION

                # 將球移出碰撞區域，防止卡住
                overlap = BALL_RADIUS - dist
                ball_pos += collision_normal * overlap

    # 邊界條件（若球飛出畫面，簡單反彈處理）
    if ball_pos.x - BALL_RADIUS < 0 or ball_pos.x + BALL_RADIUS > WIDTH:
        ball_vel.x = -ball_vel.x * FRICTION
    if ball_pos.y - BALL_RADIUS < 0 or ball_pos.y + BALL_RADIUS > HEIGHT:
        ball_vel.y = -ball_vel.y * FRICTION

    # 畫面繪製
    screen.fill((30, 30, 30))  # 背景顏色

    # 畫出六邊形（白色線條）
    hex_point_list = [(int(p.x), int(p.y)) for p in hex_points]
    pygame.draw.polygon(screen, (255, 255, 255), hex_point_list, 3)

    # 畫出球（紅色圓形）
    pygame.draw.circle(screen, (255, 0, 0), (int(ball_pos.x), int(ball_pos.y)), BALL_RADIUS)

    pygame.display.flip()

pygame.quit()
sys.exit()


pygame 2.6.1 (SDL 2.28.4, Python 3.11.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
